# Physics-Informed Neural Networks for Urban Air Pollution Modelling
### Analysis, simulation and prediction of PM$_{2.5}$ over Delhi

This notebook is the computational companion to the project report. It reads like a short paper:
each section states **what** we do, **why**, the **mathematics**, the **algorithm**, and then **interprets** the output.

**How it runs.** Everything below executes top to bottom in a few minutes on a laptop:

* the mathematical checks (autograd derivatives, exact Gaussian-puff solution, solver convergence, stability) run **live**;
* a short PINN-versus-data-only training run on the synthetic twin runs **live** (a demonstration, fewer iterations than the study);
* the headline results (4 + 6 full training runs of 6000 iterations, several hours of CPU time) are **loaded** from
  tables and figures embedded in this notebook; see the last section for how they were produced.

**Self-contained.** This is the only file needed. All code is defined in the cells below, and the data, stored results
and figures are embedded in the notebook itself, so no other project files or network access are required. It needs Python with
`numpy`, `pandas`, `scipy`, `matplotlib`, `torch` and `ipython`.

**Labels.** Every quantity is tagged as **OBSERVED** (monitor measurements), **ESTIMATED** (ERA5 reanalysis),
**CALIBRATED** (learned by the PINN) or **SYNTHETIC** (generated by our solver).

In [ ]:
import base64, copy, io, json, lzma, tarfile, time
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.stats import qmc
from torch import nn
from IPython.display import Image, display

# Frozen experimental configuration. All values used below come from this dictionary.
CONFIG = {
    "seed": 42,
    "domain": {
        "name": "Delhi NCT",
        # Local tangent-plane projection origin (degrees). x = east (km), y = north (km).
        "lat0": 28.63, "lon0": 77.15,
        # Bounding box used to select monitoring stations (degrees).
        "lat_min": 28.40, "lat_max": 28.88, "lon_min": 76.84, "lon_max": 77.35,
    },
    # Two 10-day study windows (local time, Asia/Kolkata). End is exclusive.
    "windows": {
        "winter": {"start": "2025-11-10", "end": "2025-11-20"},
        "spring": {"start": "2025-03-01", "end": "2025-03-11"},
    },
    "data": {
        "pollutant": "pm25", "units": "µg/m³", "timezone": "Asia/Kolkata",
        "valid_min": 0.0, "valid_max": 2000.0,  # physically plausible hourly PM2.5 range (µg/m³)
        "flatline_hours": 6,                    # same value for this many hours = stuck sensor
        "min_coverage": 0.75,                   # minimum fraction of valid hours per station
    },
    "split": {"train": 0.70, "val": 0.15, "test": 0.15},  # fractions of STATIONS, not observations
    "physics": {
        "lam": 5.0e-6,                          # removal coefficient λ (1/s), fixed
        "D_min": 10.0, "D_max": 2000.0, "D_init": 300.0,  # learnable eddy diffusivity D (m²/s)
        "H_min": 50.0, "H_ref": 500.0,          # mixing-height floor and reference scale (m)
    },
    "twin": {  # SYNTHETIC experiment parameters
        "start": "2024-11-12T00:00Z", "days": 3, "spinup_hours": 24, "grid_km": 1.0,
        "D_true": 300.0, "lam_true": 5.0e-6,
        "C_inflow": 60.0,                       # µg/m³ regional background on inflow boundaries
        "E_background": 0.4,                    # µg m⁻² s⁻¹ city-wide emission flux
        # [x_km, y_km, sigma_km, peak flux µg m⁻² s⁻¹]; illustrative, not an inventory
        "hotspots": [[8.0, 2.0, 4.0, 1.5], [-15.0, 10.0, 5.0, 1.2], [-2.0, -8.0, 3.0, 1.0]],
        "noise_rel": 0.05,                      # multiplicative Gaussian observation noise
        "placeholder_stations": 36,
    },
}

cfg = copy.deepcopy(CONFIG)
SEED = cfg["seed"]
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e1e0d9", "font.size": 9})
COL = {"PINN": "#2a78d6", "DataOnlyNN": "#eb6834", "IDW": "#1baf7a", "RandomForest": "#eda100", "StationMean": "#6250d6"}
print("numpy", np.__version__, "| pandas", pd.__version__, "| torch", torch.__version__, "| seed", SEED)

### Embedded data
The next cell holds every input file as one compressed, base64-encoded archive: the cleaned monitor tables, ERA5 forcing,
the synthetic-twin data, the stored results of the full training runs, and the figures. Its source is collapsed
because it contains only encoded data. The cell after it unpacks the archive into memory (nothing is written to disk).

In [ ]:
#@title Embedded data archive (encoded data only)
EMBEDDED_ARCHIVE = """
...
"""

In [ ]:
with tarfile.open(fileobj=io.BytesIO(lzma.decompress(base64.b64decode(EMBEDDED_ARCHIVE))), mode="r:") as _tar:
    FILES = {m.name: _tar.extractfile(m).read() for m in _tar.getmembers()}


def read_csv(path, **kwargs):
    """Read an embedded CSV table."""
    return pd.read_csv(io.BytesIO(FILES[path]), **kwargs)


def read_json(path):
    """Read an embedded JSON document."""
    return json.loads(FILES[path].decode("utf-8"))


def show_figure(name, width=850):
    """Display a stored figure from the full study."""
    display(Image(data=FILES[f"results/figures/{name}.png"], format="png", width=width))


print(f"{len(FILES)} embedded files, {sum(map(len, FILES.values())) / 1e6:.1f} MB unpacked")

## 1. Real-world data

**What.** Hourly PM$_{2.5}$ from regulatory monitors (CPCB, DPCC, IMD, IITM) in Delhi, obtained from the
OpenAQ open-data archive, and hourly ERA5 reanalysis wind and boundary-layer height from Open-Meteo.

**Why.** The brief requires real observations with verifiable provenance. Delhi has one of the densest regulatory networks
in India (about 45 usable monitors in a 50 km box), which is what a *spatial* model needs.

**Cleaning algorithm** (applied before this notebook; the cleaned tables are embedded above):
1. keep PM$_{2.5}$, reject units other than µg/m³;
2. drop sentinels (`-999`), negatives and values $> 2000$ µg/m³;
3. convert IST to UTC, average 15-minute values to hourly means (this also collapses duplicate timestamps);
4. remove stuck sensors (≥ 6 identical consecutive hours);
5. drop stations outside the study box or with < 75 % hourly coverage.

The winter window was planned for November 2024, but the archive holds no CPCB/DPCC files for that month,
so it was moved to the same dates in 2025.

In [ ]:
report = pd.DataFrame(read_json("results/metrics/real_cleaning.json")).T
obs = {w: read_csv(f"data/processed/real/{w}_observations.csv", parse_dates=["time"]) for w in ("winter", "spring")}
forcing = {w: read_csv(f"data/processed/real/{w}_forcing.csv", parse_dates=["time"]) for w in ("winter", "spring")}
report[["raw_pm25_records", "after_cleaning", "after_coverage", "stations", "start_utc", "end_utc"]]

In [ ]:
# OBSERVED: station locations (local km grid) coloured by window-mean PM2.5, and data availability per station-hour.
fig, axs = plt.subplots(2, 2, figsize=(10, 7), gridspec_kw=dict(width_ratios=[1, 1.6]))
for row, w in enumerate(obs):
    o = obs[w]
    st = o.groupby("location_id").agg(x=("x", "first"), y=("y", "first"), mean=("pm25", "mean"))
    sc = axs[row, 0].scatter(st.x, st.y, c=st["mean"], cmap="Blues", edgecolors="k", s=40)
    plt.colorbar(sc, ax=axs[row, 0], label="mean PM2.5 (µg/m³)")
    axs[row, 0].set(aspect="equal", xlabel="x east (km)", ylabel="y north (km)", title=f"{w}: {len(st)} monitors")
    avail = o.pivot_table(index="location_id", columns="time", values="pm25", aggfunc="size").notna()
    axs[row, 1].imshow(avail.values, aspect="auto", cmap="Greys", interpolation="nearest")
    axs[row, 1].set(xlabel="hour of window", ylabel="station (index)",
                    title=f"{w}: available station-hours (black) = {avail.values.mean():.0%}")
    axs[row, 1].grid(False)
plt.tight_layout(); plt.show()

**Interpretation.** Coverage after cleaning is above 90 % of station-hours; gaps are short and scattered, so the
data loss simply uses the hours that exist (no gap-filling, no interpolated "observations"). Concentrations differ
markedly between stations, which is the spatial signal a model must reconstruct at held-out monitors.

In [ ]:
# OBSERVED: temporal structure and how much of it is shared between stations.
fig, axs = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, w in zip(axs, obs):
    o = obs[w].assign(hour=(obs[w].time.dt.tz_convert("Asia/Kolkata").dt.hour))
    q = o.groupby("hour").pm25.quantile([0.25, 0.5, 0.75]).unstack()
    ax.fill_between(q.index, q[0.25], q[0.75], color="#cde2fb"); ax.plot(q.index, q[0.5], color=COL["PINN"])
    ax.set(xlabel="hour of day (IST)", ylabel="PM2.5 (µg/m³)", title=f"{w}: diurnal median and IQR")
plt.tight_layout(); plt.show()
for w, o in obs.items():
    c = o.pivot_table(index="time", columns="location_id", values="pm25").corr().values
    print(f"{w}: mean {o.pm25.mean():.0f} µg/m³, median inter-station correlation r = {np.nanmedian(c[np.triu_indices_from(c, 1)]):.2f}")

**Interpretation.** Both windows peak at night and bottom out in mid-afternoon — the signature of the boundary layer
collapsing at night and deepening by day, which is why the mixing height enters our source term. The median
inter-station correlation of about 0.7 says that much of the variation is shared across the city; this will matter
when we interpret the real-data results.

## 2. Mathematical model

Conservation of mass in a well-mixed layer of depth $H(t)$, with uniform wind, K-theory diffusion, linear removal and
a separable emission source, gives the 2D advection–diffusion–reaction equation

$$
\frac{\partial C}{\partial t} + u(t)\frac{\partial C}{\partial x} + v(t)\frac{\partial C}{\partial y}
= D\nabla^2 C - \lambda C + \frac{q(x,y)\,g(\tau)}{\max\{H(t),H_{\min}\}} .
$$

| Symbol | Meaning | Status |
|---|---|---|
| $C$ | PM$_{2.5}$ (µg m$^{-3}$) | observed at monitors |
| $u, v$ | 10 m wind (m s$^{-1}$) | estimated (ERA5) |
| $H$ | boundary-layer height (m), floored at $H_{\min}=50$ m | estimated (ERA5) |
| $D$ | eddy diffusivity (m$^2$ s$^{-1}$), bounded to [10, 2000] | calibrated |
| $\lambda$ | removal rate, $5\times10^{-6}$ s$^{-1}$ | fixed |
| $q(x,y)\ge 0$, $g(\tau)$ | emission flux field and 24-h profile (mean 1) | calibrated |

**Assumptions:** well-mixed layer (2D), spatially uniform wind, constant isotropic $D$, linear removal,
separable emissions, unknown inflow (weak outflow condition only), no entrainment term.

### Dimensionless groups
Scaling $x' = (x-x_c)/L$, $t' = 2t/T-1$, $C' = C/C_{\mathrm{ref}}$ gives coefficients $U=uT/2L$, $K=DT/2L^2$,
$\Lambda=\lambda T/2$. The physical groups are the Péclet number $\mathrm{Pe}=\bar u\ell/D$ and the Damköhler number
$\mathrm{Da}=\lambda\ell/\bar u$. We compute them from the real forcing:

In [ ]:
EARTH_RADIUS_KM = 6371.0


def project_to_local(lat, lon, lat0, lon0):
    """Equirectangular projection to km about (lat0, lon0).

    Distortion over a 50 km domain is < 0.1 %, far below other model errors.
    """
    lat, lon = np.asarray(lat, float), np.asarray(lon, float)
    x = EARTH_RADIUS_KM * np.cos(np.deg2rad(lat0)) * np.deg2rad(lon - lon0)
    y = EARTH_RADIUS_KM * np.deg2rad(lat - lat0)
    return x, y


def domain_km(dom):
    x0, y0 = project_to_local([dom["lat_min"]], [dom["lon_min"]], dom["lat0"], dom["lon0"])
    x1, y1 = project_to_local([dom["lat_max"]], [dom["lon_max"]], dom["lat0"], dom["lon0"])
    return float(x0[0]), float(x1[0]), float(y0[0]), float(y1[0])

In [ ]:
xmin, xmax, ymin, ymax = domain_km(cfg["domain"])
ell = max(xmax - xmin, ymax - ymin) * 1e3       # m
lam = cfg["physics"]["lam"]
for w, f in forcing.items():
    ubar = f.wind_speed_10m.mean()
    print(f"{w}: mean wind {ubar:.2f} m/s | advection time l/u = {ell / ubar / 3600:.1f} h | "
          f"Pe = {ubar * ell / 1000:.0f}–{ubar * ell / 100:.0f} (D = 1000–100 m²/s) | Da = {lam * ell / ubar:.2f} | "
          f"H range {f.blh.min():.0f}–{f.blh.max():.0f} m")

**Interpretation.** $\mathrm{Pe}\gg 1$: transport is dominated by the wind, and diffusion is a small correction —
so we should *expect* $D$ to be poorly identifiable. $\mathrm{Da}\ll 1$: removal is slow compared with transit.
Air crosses the city in 7–11 hours, so hourly data can, in principle, see transport between stations. The ERA5 mixing
height falls to 15–25 m on calm nights in these windows (and to ~10 m in the twin's forcing), which motivates the floor $H_{\min}=50$ m.

## 3. Mathematical analysis, checked numerically

### 3.1 Fourier stability and equilibrium
For constant coefficients, $C=\hat c\,e^{i\mathbf k\cdot\mathbf x+\sigma t}$ gives
$\sigma = -i\,\mathbf u\cdot\mathbf k - D|\mathbf k|^2 - \lambda$, so $\operatorname{Re}\sigma = -(D|\mathbf k|^2+\lambda) < 0$:
every mode decays (advection only rotates its phase). For a uniform constant source $S_0$ the uniform solution
$C(t) = S_0/\lambda + (C_0 - S_0/\lambda)e^{-\lambda t}$ tends to the stable equilibrium $C^* = S_0/\lambda$.

**Reference solver.** The finite-difference solver used throughout; its discretisation is analysed in §3.3.

In [ ]:
"""Reference finite-difference solver for the 2D ADR equation.

Used (i) to generate the synthetic-twin data set, (ii) as the "numerical
simulation" component of the report, and (iii) as an independent check of the
physics implemented in the PINN.

Discretisation (method of lines on a uniform cell-centred grid):
  * advection  — third-order upwind-biased differences (low numerical diffusion);
  * diffusion  — second-order central differences;
  * time       — strong-stability-preserving Runge-Kutta 3 (SSP-RK3);
  * boundaries — inflow faces: Dirichlet C = C_bc(t); outflow faces: zero gradient.
Stability: dt is limited by the advective CFL number and the diffusive number
D dt (1/dx^2 + 1/dy^2) <= 1/4 (see `stable_dt`; von Neumann analysis in report).
"""


G = 2  # ghost cells per side


@dataclass
class Grid:
    x_min: float
    x_max: float
    y_min: float
    y_max: float
    nx: int
    ny: int
    x: np.ndarray = field(init=False)
    y: np.ndarray = field(init=False)

    def __post_init__(self):
        self.x = np.linspace(self.x_min, self.x_max, self.nx)
        self.y = np.linspace(self.y_min, self.y_max, self.ny)
        self.X, self.Y = np.meshgrid(self.x, self.y, indexing="ij")

    @property
    def dx(self):
        return self.x[1] - self.x[0]

    @property
    def dy(self):
        return self.y[1] - self.y[0]

    @property
    def shape(self):
        return (self.nx, self.ny)


def stable_dt(grid: Grid, umax, vmax, D, cfl=0.5, diff_number=0.25) -> float:
    adv = abs(umax) / grid.dx + abs(vmax) / grid.dy
    dif = D * (1 / grid.dx**2 + 1 / grid.dy**2)
    limits = [cfl / adv if adv > 0 else np.inf, diff_number / dif if dif > 0 else np.inf]
    return float(min(limits))


def _pad(C, u, v, cb):
    """Fill ghost cells: Dirichlet on inflow faces, zero-gradient on outflow faces."""
    P = np.pad(C, G, mode="edge")
    if u > 0:
        P[:G, :] = cb
    elif u < 0:
        P[-G:, :] = cb
    if v > 0:
        P[:, :G] = cb
    elif v < 0:
        P[:, -G:] = cb
    return P


def _upwind3(P, vel, h, axis):
    """Third-order upwind-biased first derivative on the interior of padded P."""
    s = lambda k: np.roll(P, -k, axis=axis)  # s(k)[i] = P[i + k]
    if vel >= 0:
        d = (2 * s(1) + 3 * P - 6 * s(-1) + s(-2)) / (6 * h)
    else:
        d = (-s(2) + 6 * s(1) - 3 * P - 2 * s(-1)) / (6 * h)
    return d[G:-G, G:-G]


def _rhs(C, t, grid, u, v, D, lam, S, C_bc):
    ut, vt, cb = u(t), v(t), C_bc(t)
    P = _pad(C, ut, vt, cb)
    adv = ut * _upwind3(P, ut, grid.dx, 0) + vt * _upwind3(P, vt, grid.dy, 1)
    lap = (P[G + 1:-G + 1 or None, G:-G] - 2 * C + P[G - 1:-G - 1, G:-G]) / grid.dx**2 \
        + (P[G:-G, G + 1:-G + 1 or None] - 2 * C + P[G:-G, G - 1:-G - 1]) / grid.dy**2
    return -adv + D * lap - lam * C + S(grid.X, grid.Y, t)


def solve_adr(grid, C0, t_end, u, v, D, lam, S, C_bc, save_every=3600.0, t0=0.0,
              umax=None, vmax=None, cfl=0.5):
    """Integrate from t0 to t0 + t_end. u, v, C_bc are functions of t; S of (X, Y, t).

    Returns dict with 't' (saved times) and 'C' (array [n_saved, nx, ny]); the
    final state is always saved.
    """
    if umax is None or vmax is None:
        ts = np.linspace(t0, t0 + t_end, 200)
        umax = max(abs(u(s)) for s in ts) if umax is None else umax
        vmax = max(abs(v(s)) for s in ts) if vmax is None else vmax
    dt_max = stable_dt(grid, umax, vmax, D, cfl=cfl)
    if not np.isfinite(dt_max):
        dt_max = t_end / 100

    f = lambda C, t: _rhs(C, t, grid, u, v, D, lam, S, C_bc)
    C, t = np.array(C0, dtype=float), t0
    times, states = [], []
    next_save = t0 + save_every if save_every else np.inf
    if save_every:
        times.append(t), states.append(C.copy())
    t_stop = t0 + t_end
    while t < t_stop - 1e-9:
        dt = min(dt_max, t_stop - t, next_save - t)
        k1 = C + dt * f(C, t)
        k2 = 0.75 * C + 0.25 * (k1 + dt * f(k1, t + dt))
        C = C / 3 + 2 / 3 * (k2 + dt * f(k2, t + 0.5 * dt))
        t += dt
        if save_every and abs(t - next_save) < 1e-9:
            times.append(t), states.append(C.copy())
            next_save += save_every
    if not times or times[-1] != t:
        times.append(t), states.append(C.copy())
    return {"t": np.array(times), "C": np.array(states)}

In [ ]:
k = np.logspace(-6, -2.5, 200)                       # wavenumber (1/m)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.2))
for D in (100, 300, 1000):
    axs[0].loglog(2 * np.pi / k / 1e3, (D * k**2 + lam) * 3600, label=f"D = {D} m²/s")
axs[0].axhline(lam * 3600, color="k", ls=":", label="λ alone")
axs[0].set(xlabel="wavelength (km)", ylabel="decay rate −Re σ (1/h)", title="Every Fourier mode decays"); axs[0].legend()

# Equilibrium: uniform source and removal with the reference solver vs the exact solution.
g = Grid(-1e4, 1e4, -1e4, 1e4, 21, 21)
S0, lam_demo = 2e-3, 1e-4
exact = lambda t: S0 / lam_demo * (1 - np.exp(-lam_demo * t))
sol = solve_adr(g, np.zeros(g.shape), 50_000, lambda t: 1.5, lambda t: -0.5, 100.0, lam_demo,
                lambda X, Y, t: S0, exact, save_every=2500.0)
axs[1].plot(sol["t"] / 3600, sol["C"].mean(axis=(1, 2)), "o", ms=4, color=COL["PINN"], label="solver (domain mean)")
axs[1].plot(sol["t"] / 3600, exact(sol["t"]), "k-", label="exact")
axs[1].axhline(S0 / lam_demo, color="k", ls=":", label="C* = S₀/λ")
axs[1].set(xlabel="time (h)", ylabel="C (µg/m³)", title="Approach to equilibrium"); axs[1].legend()
plt.tight_layout(); plt.show()
print("max relative error vs exact:", np.max(np.abs(sol["C"].mean(axis=(1, 2)) - exact(sol["t"]))[1:] / exact(sol["t"][1:])))

**Interpretation.** Short wavelengths are damped fastest by diffusion; long ones decay at the removal rate $\lambda$.
The solver reproduces the exact approach to $C^*$, confirming the reaction and source terms and the boundary handling.

### 3.2 Automatic differentiation and the PDE residual
The PINN needs $\partial_t C$, $\partial_x C$, $\partial_y C$ and $\nabla^2 C$ of a neural network. We check our autograd
implementation (defined in the next cell) against two independent references:
a manufactured function with known derivatives, and the **exact Gaussian puff**

$$C=\frac{M}{4\pi Dt}\exp\!\Big(-\frac{|\mathbf x-\mathbf x_0-\mathbf u t|^2}{4Dt}-\lambda t\Big),$$

which solves the PDE with $S=0$, so its residual must vanish.

In [ ]:
"""Advection-diffusion-reaction (ADR) operator for the PINN.

Governing equation (mixing-layer-averaged PM2.5, 2D + time):

    C_t + u(t) C_x + v(t) C_y = D (C_xx + C_yy) - lam C + S(x, y, t)

The residual  r = C_t + u C_x + v C_y - D (C_xx + C_yy) + lam C - S  is computed
with automatic differentiation. Coefficients may be scalars or per-point tensors,
and must be expressed in the same units as the coordinates used for X.
"""


def _grad(f: torch.Tensor, X: torch.Tensor) -> torch.Tensor:
    # A derivative that is constant in X (e.g. of a linear field) is not connected
    # to X in the graph; its gradient is then identically zero.
    if not f.requires_grad:
        return torch.zeros_like(X)
    g = torch.autograd.grad(f, X, grad_outputs=torch.ones_like(f), create_graph=True, allow_unused=True)[0]
    return torch.zeros_like(X) if g is None else g


def gradients(C: torch.Tensor, X: torch.Tensor) -> dict:
    """First derivatives and the two pure second space derivatives of C(X), X = (x, y, t)."""
    dC = _grad(C, X)
    C_x, C_y, C_t = dC[:, 0], dC[:, 1], dC[:, 2]
    C_xx = _grad(C_x, X)[:, 0]
    C_yy = _grad(C_y, X)[:, 1]
    return {"x": C_x, "y": C_y, "t": C_t, "xx": C_xx, "yy": C_yy}


def adr_residual(C, X, u, v, D, lam, S) -> torch.Tensor:
    """Pointwise PDE residual r(x, y, t); zero for an exact solution."""
    d = gradients(C, X)
    return d["t"] + u * d["x"] + v * d["y"] - D * (d["xx"] + d["yy"]) + lam * C - S


def nondimensional_coefficients(u, v, D, lam, S, L, T, C_ref) -> dict:
    """Coefficients of the PDE in variables x' = x/L, y' = y/L, t' = t/T, C' = C/C_ref.

    Dividing the physical PDE by C_ref/T gives
        C'_t' + (uT/L) C'_x' + (vT/L) C'_y' = (DT/L^2) lap' C' - (lam T) C' + S T / C_ref,
    i.e. the groups uT/L (advective Courant-like number), DT/L^2 (= (uT/L)/Pe)
    and lam T (Damkohler-like number).
    """
    return {"u": u * T / L, "v": v * T / L, "D": D * T / L**2, "lam": lam * T, "S": S * T / C_ref}


def gaussian_puff(x, y, t, M, x0, y0, u, v, D, lam):
    """Exact solution for an instantaneous release of mass M (per unit depth) at t = 0.

        C = M / (4 pi D t) * exp(-[(x - x0 - u t)^2 + (y - y0 - v t)^2] / (4 D t) - lam t)

    Works with NumPy arrays or torch tensors.
    """
    lib = torch if isinstance(t, torch.Tensor) else np
    r2 = (x - x0 - u * t) ** 2 + (y - y0 - v * t) ** 2
    return M / (4 * np.pi * D * t) * lib.exp(-r2 / (4 * D * t) - lam * t)

In [ ]:
X = torch.rand(500, 3, dtype=torch.float64, requires_grad=True)
x, y, t = X[:, 0], X[:, 1], X[:, 2]
C = torch.sin(2 * x) * torch.cos(3 * y) * torch.exp(-t)
d = gradients(C, X)
lap_exact = -13 * torch.sin(2 * x) * torch.cos(3 * y) * torch.exp(-t)
print("max |autograd Laplacian − analytic| =", (d["xx"] + d["yy"] - lap_exact).abs().max().item())

Xp = (torch.rand(500, 3, dtype=torch.float64) * torch.tensor([4.0, 4.0, 1.0]) + torch.tensor([-2.0, -2.0, 0.5])).requires_grad_(True)
u_, v_, D_, lam_ = 0.8, 0.3, 0.1, 0.2
Cp = gaussian_puff(Xp[:, 0], Xp[:, 1], Xp[:, 2], M=1.0, x0=0.0, y0=0.0, u=u_, v=v_, D=D_, lam=lam_)
print("max |PDE residual| of the exact puff =", adr_residual(Cp, Xp, u_, v_, D_, lam_, torch.zeros(500)).abs().max().item())

**Interpretation.** Both errors are at the level of floating-point round-off ($\sim 10^{-14}$), so the residual operator
used in training is exactly the PDE we wrote down.

### 3.3 Reference solver: discretisation, stability, convergence
Method of lines: third-order upwind-biased advection, second-order central diffusion, SSP-RK3 in time. Von Neumann
analysis bounds the time step by the Courant number $\Delta t(|u|/\Delta x+|v|/\Delta y)\le 0.5$ and the diffusion number
$D\Delta t(\Delta x^{-2}+\Delta y^{-2})\le 0.25$. We verify convergence to the exact puff under grid refinement:

In [ ]:
def puff_error(n):
    g = Grid(-15e3, 15e3, -15e3, 15e3, n, n)
    p = lambda X, Y, t: gaussian_puff(X, Y, t, M=1e10, x0=-12e3, y0=-8e3, u=2.0, v=1.0, D=400.0, lam=1e-5)
    out = solve_adr(g, p(g.X, g.Y, 3600.0), 5400.0, lambda t: 2.0, lambda t: 1.0, 400.0, 1e-5,
                    lambda X, Y, t: 0.0, lambda t: 0.0, save_every=None, t0=3600.0)
    e = p(g.X, g.Y, 9000.0)
    return g.dx, np.linalg.norm(out["C"][-1] - e) / np.linalg.norm(e)

rows = [puff_error(n) for n in (41, 81, 161)]
conv = pd.DataFrame(rows, columns=["dx (m)", "relative L2 error"])
conv["observed order"] = [np.nan] + [np.log2(rows[i][1] / rows[i + 1][1]) for i in range(2)]
conv

**Interpretation.** Halving the grid spacing reduces the error by roughly an order of magnitude (observed orders between 2.6
and 3.7, consistent with third-order advection and second-order diffusion). The solver is therefore accurate enough to serve
as the "truth" of the synthetic twin.

## 4. The PINN

**Architecture** (implemented in the cells below):
* concentration network $(x',y',t') \mapsto \hat C' = \mathrm{softplus}(\mathrm{MLP}_{5\times64}^{\tanh})$ — softplus keeps
  $C\ge 0$; $\tanh$ keeps second derivatives non-zero (ReLU would make $\nabla^2\hat C = 0$ almost everywhere);
* source model $S' = q(x',y')\,g(\tau)/h(t)$ with $q$ a small softplus MLP ($2\times 32$) and $g$ 24 hourly values with mean 1;
* bounded diffusivity $D = D_{\min} + (D_{\max}-D_{\min})\,\sigma(\theta_D)$.

Meteorology is **not** a network input, so $\partial \hat C/\partial t$ is a true partial derivative; the wind and mixing height
enter only through the PDE coefficients.

**Loss.**
$$\mathcal L = w_d\underbrace{\tfrac1{N_d}\textstyle\sum(\hat C'_j-C'_j)^2}_{\text{stations}}
+ w_p\underbrace{\tfrac1{N_c}\textstyle\sum r_k^2}_{\text{PDE residual}}
+ w_b\underbrace{\tfrac1{N_b}\textstyle\sum_{\mathbf u\cdot\mathbf n>0}(\nabla\hat C'\cdot\mathbf n)^2}_{\text{outflow}}
+ w_s\underbrace{\big(\overline{|\nabla q|^2} + \text{curvature of } g\big)}_{\text{source regulariser}}$$

The source regulariser is essential: an unrestricted source could absorb any residual and make the physics term vacuous.

**Algorithm.**
1. Sample 4096 Latin-hypercube collocation points and 512 boundary points in $[-1,1]^3$ (resampled every 250 steps).
2. Each step: evaluate the four loss terms (derivatives by autograd), take an Adam step; learning rate $10^{-3}\to10^{-4}$.
3. Every 100 steps, score the **validation stations** and keep the best weights (they never contribute gradients).

The **data-only network** is the identical model with $w_p=w_b=w_s=0$ — the controlled comparison.

**Implementation: model.** Scaling, interpolated forcing, networks, source model and bounded diffusivity.

In [ ]:
IST_OFFSET_H = 5.5


@dataclass
class Scales:
    x_c: float  # m
    y_c: float  # m
    L: float  # m, half-width of the domain
    t_start: pd.Timestamp  # UTC, t' = -1
    T: float  # s, window length, t' = +1 at t_start + T
    C_ref: float  # µg/m³

    def inputs(self, x_m, y_m, t_rel_s) -> torch.Tensor:
        X = np.stack([(np.asarray(x_m) - self.x_c) / self.L,
                      (np.asarray(y_m) - self.y_c) / self.L,
                      2 * np.asarray(t_rel_s) / self.T - 1], axis=1)
        return torch.as_tensor(X, dtype=torch.get_default_dtype())

    def t_rel(self, t_nd: torch.Tensor) -> torch.Tensor:
        return (t_nd + 1) * self.T / 2

    def local_hour(self, t_nd: torch.Tensor) -> torch.Tensor:
        h0 = self.t_start.hour + self.t_start.minute / 60 + IST_OFFSET_H
        return torch.remainder(h0 + self.t_rel(t_nd) / 3600.0, 24.0)


class Forcing(nn.Module):
    """Hourly ERA5 wind (m/s) and boundary-layer height, linearly interpolated in time."""

    def __init__(self, t_rel_s, u, v, H, H_min=50.0, H_ref=500.0):
        super().__init__()
        f = lambda a: torch.tensor(np.array(a, dtype=float), dtype=torch.get_default_dtype())
        self.register_buffer("t", f(t_rel_s))
        self.register_buffer("u", f(u))
        self.register_buffer("v", f(v))
        self.register_buffer("h", f(np.maximum(H, H_min) / H_ref))

    def at(self, t_rel_s: torch.Tensor):
        t = t_rel_s.detach().to(self.t.dtype).clamp(self.t[0], self.t[-1])
        i = torch.searchsorted(self.t, t, right=True).clamp(1, len(self.t) - 1)
        w = (t - self.t[i - 1]) / (self.t[i] - self.t[i - 1])
        lerp = lambda a: a[i - 1] + w * (a[i] - a[i - 1])
        return lerp(self.u), lerp(self.v), lerp(self.h)


def mlp(n_in, n_out, width, depth) -> nn.Sequential:
    layers, d = [], n_in
    for _ in range(depth):
        lin = nn.Linear(d, width)
        nn.init.xavier_normal_(lin.weight)
        nn.init.zeros_(lin.bias)
        layers += [lin, nn.Tanh()]
        d = width
    out = nn.Linear(d, n_out)
    nn.init.xavier_normal_(out.weight)
    nn.init.zeros_(out.bias)
    return nn.Sequential(*layers, out)


class ConcentrationNet(nn.Module):
    """(x', y', t') -> C' >= 0 (softplus output). tanh keeps second derivatives non-zero."""

    def __init__(self, width=64, depth=5):
        super().__init__()
        self.body = mlp(3, 1, width, depth)

    def forward(self, X):
        return nn.functional.softplus(self.body(X))[:, 0]


class SourceModel(nn.Module):
    """S'(x', y', t') = q(x', y') g(tau) / h(t): spatial emission x diurnal profile / mixing height."""

    def __init__(self, scales: Scales, forcing: Forcing, width=32, depth=2):
        super().__init__()
        self.scales, self.forcing = scales, forcing
        self.q_net = mlp(2, 1, width, depth)
        self.g_raw = nn.Parameter(torch.zeros(24))

    def q(self, X):
        return nn.functional.softplus(self.q_net(X[:, :2]))[:, 0]

    def g_nodes(self):
        g = nn.functional.softplus(self.g_raw)
        return g / g.mean()

    def g(self, hours):
        """Periodic piecewise-linear interpolation of the 24 hourly nodes."""
        nodes = self.g_nodes()
        h = torch.remainder(hours, 24.0)
        i0 = torch.floor(h).long() % 24
        w = h - torch.floor(h)
        return (1 - w) * nodes[i0] + w * nodes[(i0 + 1) % 24]

    def forward(self, X):
        _, _, h = self.forcing.at(self.scales.t_rel(X[:, 2]))
        return self.q(X) * self.g(self.scales.local_hour(X[:, 2])) / h


class PINN(nn.Module):
    def __init__(self, scales: Scales, forcing: Forcing, D_min, D_max, D_init, lam,
                 width=64, depth=5, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.scales, self.forcing, self.lam = scales, forcing, float(lam)
        self.D_min, self.D_max = float(D_min), float(D_max)
        self.net = ConcentrationNet(width, depth)
        self.source = SourceModel(scales, forcing)
        p = (D_init - D_min) / (D_max - D_min)
        self.D_raw = nn.Parameter(torch.tensor(float(np.log(p / (1 - p)))))

    def D(self):
        """Eddy diffusivity (m²/s), constrained to [D_min, D_max] by a sigmoid."""
        return self.D_min + (self.D_max - self.D_min) * torch.sigmoid(self.D_raw)

    def forward(self, X):
        return self.net(X)

    def residual(self, X):
        """Dimensionless PDE residual at collocation points X (requires_grad=True)."""
        C = self.net(X)
        u, v, _ = self.forcing.at(self.scales.t_rel(X[:, 2]))
        k = nondimensional_coefficients(u=u, v=v, D=self.D(), lam=self.lam, S=0.0,
                                        L=self.scales.L, T=self.scales.T / 2, C_ref=1.0)
        return adr_residual(C, X, k["u"], k["v"], k["D"], k["lam"], self.source(X))

**Implementation: loss terms.**

In [ ]:
"""Loss components of the PINN (all in dimensionless units).

L = w_d L_data + w_p L_phys + w_b L_bc + w_i L_ic + w_s L_src
"""


def data_loss(model, X, c):
    """Mean squared misfit at observation points (training stations only)."""
    return torch.mean((model(X) - c) ** 2)


def physics_loss(model, Xc):
    """Mean squared PDE residual at collocation points."""
    return torch.mean(model.residual(Xc) ** 2)


def boundary_loss(model, Xb, normals, c_bc=None):
    """Outflow faces (u·n > 0): zero normal gradient. Inflow faces: C' = c_bc if given.

    On real data the inflow concentration is unknown, so c_bc=None and only the
    weak outflow condition is imposed.
    """
    C = model(Xb)
    dC = _grad(C, Xb)[:, :2]
    u, v, _ = model.forcing.at(model.scales.t_rel(Xb[:, 2]))
    un = u * normals[:, 0] + v * normals[:, 1]
    out = un > 0
    terms = (torch.sum(dC * normals, dim=1) ** 2) * out
    if c_bc is not None:
        terms = terms + ((C - c_bc) ** 2) * (~out)
    return terms.mean()


def initial_loss(model, X0, c0):
    """Misfit to a known initial field at t' = -1 (synthetic twin only)."""
    return torch.mean((model(X0) - c0) ** 2)


def source_regulariser(model, Xc):
    """Smoothness of the emission field q and of the diurnal profile g.

    Without it the source can absorb any residual and the physics loss becomes vacuous.
    """
    Xs = Xc.detach().clone().requires_grad_(True)
    q = model.source.q(Xs)
    dq = _grad(q, Xs)[:, :2]
    g = model.source.g_nodes()
    curv = g.roll(-1) - 2 * g + g.roll(1)
    return torch.mean(dq ** 2) + torch.mean(curv ** 2)

**Implementation: collocation and boundary sampling.**

In [ ]:
def sample_collocation(n, seed=0) -> torch.Tensor:
    """Latin hypercube sample over (x', y', t') in [-1, 1]^3."""
    X = 2 * qmc.LatinHypercube(d=3, seed=seed).random(n) - 1
    return torch.as_tensor(X, dtype=torch.get_default_dtype())


def sample_boundary(n, seed=0):
    """Points on the four lateral faces x' = ±1, y' = ±1 with outward unit normals."""
    rng = np.random.default_rng(seed)
    face = rng.integers(0, 4, n)
    s, t = rng.uniform(-1, 1, n), rng.uniform(-1, 1, n)
    X = np.empty((n, 3))
    N = np.zeros((n, 2))
    sign = np.where(face % 2 == 0, -1.0, 1.0)
    on_x = face < 2
    X[:, 0] = np.where(on_x, sign, s)
    X[:, 1] = np.where(on_x, s, sign)
    X[:, 2] = t
    N[on_x, 0] = sign[on_x]
    N[~on_x, 1] = sign[~on_x]
    f = lambda a: torch.as_tensor(a, dtype=torch.get_default_dtype())
    return f(X), f(N)

**Implementation: training loop.**

In [ ]:
@dataclass
class TrainConfig:
    n_iters: int = 5000
    lr: float = 1e-3
    lr_final: float = 1e-4  # exponential decay from lr to lr_final
    w_d: float = 1.0
    w_p: float = 1.0
    w_b: float = 0.1
    w_i: float = 0.0
    w_s: float = 1e-3
    n_colloc: int = 4096
    n_bc: int = 512
    resample_every: int = 250
    log_every: int = 100
    seed: int = 0
    # Adaptive physics weight (gradient-norm balancing, after Wang, Teng & Perdikaris 2021):
    # w_p <- (1 - a) w_p + a * w_p0 * mean|grad L_data| / mean|grad L_phys|, over the network weights.
    adaptive: bool = False
    adapt_every: int = 100
    adapt_alpha: float = 0.1


def train(model, X_obs, c_obs, cfg: TrainConfig, val=None, ic=None, c_bc=None):
    """Minimise the weighted loss with Adam.

    X_obs, c_obs: dimensionless training observations. val: optional (X, c) used to
    keep the best weights (early-stopping selection, never for gradient steps).
    ic: optional (X0, c0) initial field. c_bc: optional inflow value (dimensionless).
    Returns the logged history; the model is left at the best-validation weights.
    """
    torch.manual_seed(cfg.seed)
    physics = cfg.w_p > 0 or cfg.w_b > 0 or cfg.w_i > 0 or cfg.w_s > 0
    params = model.parameters() if physics else model.net.parameters()
    opt = torch.optim.Adam(params, lr=cfg.lr)
    gamma = (cfg.lr_final / cfg.lr) ** (1 / max(cfg.n_iters, 1))
    sched = torch.optim.lr_scheduler.ExponentialLR(opt, gamma)

    history, best = [], (np.inf, None)
    w_p = cfg.w_p
    shared = list(model.net.parameters())
    for it in range(cfg.n_iters + 1):
        if it % cfg.resample_every == 0:
            Xc = sample_collocation(cfg.n_colloc, seed=cfg.seed + it).requires_grad_(True)
            Xb, nb = sample_boundary(cfg.n_bc, seed=cfg.seed + it)
            Xb.requires_grad_(True)

        parts = {"data": data_loss(model, X_obs, c_obs)}
        if cfg.w_p > 0:
            parts["physics"] = physics_loss(model, Xc)
        if cfg.w_b > 0:
            parts["boundary"] = boundary_loss(model, Xb, nb, c_bc)
        if cfg.w_i > 0 and ic is not None:
            parts["initial"] = initial_loss(model, *ic)
        if cfg.w_s > 0:
            parts["source_reg"] = source_regulariser(model, Xc)
        if cfg.adaptive and "physics" in parts and it > 0 and it % cfg.adapt_every == 0:
            g_d = _flat_grad(parts["data"], shared)
            g_p = _flat_grad(parts["physics"], shared)
            w_hat = cfg.w_p * g_d.abs().mean() / (g_p.abs().mean() + 1e-12)
            w_p = (1 - cfg.adapt_alpha) * w_p + cfg.adapt_alpha * w_hat.item()
        w = {"data": cfg.w_d, "physics": w_p, "boundary": cfg.w_b, "initial": cfg.w_i, "source_reg": cfg.w_s}
        total = sum(w[k] * v for k, v in parts.items())

        if it % cfg.log_every == 0 or it == cfg.n_iters:
            rec = {"iter": it, "total": total.item(), "w_p": w_p, **{k: v.item() for k, v in parts.items()},
                   "D": model.D().item() if hasattr(model, "D") else float("nan")}
            if val is not None:
                with torch.no_grad():
                    rec["val"] = torch.mean((model(val[0]) - val[1]) ** 2).item()
                if rec["val"] < best[0]:
                    best = (rec["val"], copy.deepcopy(model.state_dict()))
            history.append(rec)
        if it == cfg.n_iters:
            break
        opt.zero_grad()
        total.backward()
        opt.step()
        sched.step()

    if best[1] is not None:
        model.load_state_dict(best[1])
    return history


def _flat_grad(loss, params):
    gs = torch.autograd.grad(loss, params, retain_graph=True, allow_unused=True)
    return torch.cat([g.reshape(-1) for g in gs if g is not None])

### 4.1 Synthetic twin
To know whether the physics *can* help we need a case with a known answer. The reference solver, driven by the **real ERA5
wind and mixing height** of 12–15 November 2024 and an illustrative emission field, produces the truth; it is sampled at the
**45 real monitor locations** with 5 % noise (all SYNTHETIC).

In [ ]:
def _colocated_groups(xy: np.ndarray, min_sep_km: float) -> np.ndarray:
    """Single-linkage groups: stations closer than min_sep_km share a group label."""
    n = len(xy)
    parent = np.arange(n)

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    d = np.linalg.norm(xy[:, None, :] - xy[None, :, :], axis=-1)
    for i, j in zip(*np.where(np.triu(d < min_sep_km, k=1))):
        parent[find(i)] = find(j)
    return np.array([find(i) for i in range(n)])


def split_stations(stations: pd.DataFrame, train, val, test, seed, min_sep_km=1.0) -> dict:
    """Split STATIONS (not observations) into train/val/test sets of location_id.

    Co-located stations (closer than ``min_sep_km``) are kept in one subset so a
    test station never has a near-duplicate in training (spatial leakage).
    """
    assert abs(train + val + test - 1.0) < 1e-9
    ids = stations["location_id"].to_numpy()
    groups = _colocated_groups(stations[["x", "y"]].to_numpy(float), min_sep_km)
    uniq = np.unique(groups)
    np.random.default_rng(seed).shuffle(uniq)

    n = len(ids)
    targets = {"test": max(1, round(test * n)), "val": max(1, round(val * n))}
    out = {"train": set(), "val": set(), "test": set()}
    for g in uniq:
        members = {int(i) for i in ids[groups == g]}
        for part in ("test", "val"):
            if len(out[part]) < targets[part]:
                out[part] |= members
                break
        else:
            out["train"] |= members
    return out


def assert_no_station_leakage(train, test):
    """Held-out stations must never appear in the training table."""
    shared = set(train["location_id"]) & set(test["location_id"])
    assert not shared, f"stations in both train and test: {sorted(shared)}"


def load_twin():
    cfg = copy.deepcopy(CONFIG)
    obs = read_csv("data/processed/twin/observations.csv", parse_dates=["time"])
    forcing = read_csv("data/processed/twin/forcing.csv", parse_dates=["time"])
    meta = read_json("data/processed/twin/meta.json")
    truth = dict(np.load(io.BytesIO(FILES["data/processed/twin/truth.npz"])))
    return cfg, obs, forcing, meta, truth


def twin_setup(seed=None):
    """Data, station split, scales and forcing exactly as used for training (shared with figures)."""
    cfg, obs, forcing, meta, truth = load_twin()
    if seed is not None:
        cfg["seed"] = seed  # changes both the station split and the network initialisation
    ph = cfg["physics"]
    stations = obs.drop_duplicates("location_id")[["location_id", "x", "y"]]
    split = split_stations(stations, **{k: cfg["split"][k] for k in ("train", "val", "test")}, seed=cfg["seed"])
    part = {k: obs[obs.location_id.isin(v)].reset_index(drop=True) for k, v in split.items()}
    assert_no_station_leakage(part["train"], part["test"])
    assert_no_station_leakage(part["val"], part["test"])

    xmin, xmax, ymin, ymax = [v * 1e3 for v in meta["domain_km"]]
    sc = Scales(x_c=(xmin + xmax) / 2, y_c=(ymin + ymax) / 2, L=max(xmax - xmin, ymax - ymin) / 2,
                t_start=pd.Timestamp(meta["start"]), T=float(obs.t_rel.max()),
                C_ref=float(part["train"].pm25.mean()))
    fz = Forcing(forcing.t_rel, forcing.u, forcing.v, forcing.blh, H_min=ph["H_min"], H_ref=ph["H_ref"])
    return dict(cfg=cfg, obs=obs, forcing=forcing, meta=meta, truth=truth, split=split, part=part, sc=sc, fz=fz)


def twin_model(s):
    ph = s["cfg"]["physics"]
    return PINN(s["sc"], s["fz"], D_min=ph["D_min"], D_max=ph["D_max"], D_init=ph["D_init"], lam=ph["lam"],
                seed=s["cfg"]["seed"])

In [ ]:
tw = twin_setup(SEED)
meta, truth = tw["meta"], tw["truth"]
print({k: meta[k] for k in ("kind", "station_layout", "n_stations", "D_true", "lam_true", "noise_rel")})
print("split sizes (stations):", {k: len(v) for k, v in tw["split"].items()})

k = int(np.argmax(truth["C"].reshape(len(truth["t_rel"]), -1).std(1)))
fig, ax = plt.subplots(figsize=(5, 4.2))
im = ax.pcolormesh(truth["x"] / 1e3, truth["y"] / 1e3, truth["C"][k].T, cmap="Blues", shading="auto")
st = tw["obs"].drop_duplicates("location_id")
ax.scatter(st.x, st.y, s=10, c="k")
plt.colorbar(im, label="PM2.5 (µg/m³)"); ax.set(aspect="equal", xlabel="x (km)", ylabel="y (km)",
                                                 title=f"SYNTHETIC truth, hour {k} of the window")
plt.show()

### 4.2 Live demonstration: data-only network vs PINN
A shortened run (1500 iterations instead of 6000; seed 42 split, $w_p=0.01$). The full study's numbers follow in §4.3.

In [ ]:
"""Point-wise error metrics (concentrations in µg/m³)."""


def mae(y, p):
    return float(np.mean(np.abs(np.asarray(p) - np.asarray(y))))


def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(p) - np.asarray(y)) ** 2)))


def r2(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return float(1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2))


def evaluate(y, p) -> dict:
    """MAE, RMSE, R², mean bias (p - y) and Pearson r over pairs where both are finite."""
    y, p = np.asarray(y, float), np.asarray(p, float)
    ok = np.isfinite(y) & np.isfinite(p)
    y, p = y[ok], p[ok]
    return {
        "n": int(ok.sum()),
        "mae": mae(y, p),
        "rmse": rmse(y, p),
        "r2": r2(y, p),
        "bias": float(np.mean(p - y)),
        "pearson_r": float(np.corrcoef(y, p)[0, 1]) if len(y) > 1 else float("nan"),
    }

In [ ]:
sc, fz, part = tw["sc"], tw["fz"], tw["part"]
tens = lambda d: (sc.inputs(d.x * 1e3, d.y * 1e3, d.t_rel), torch.as_tensor(d.pm25.to_numpy() / sc.C_ref, dtype=torch.float32))
Xtr, ctr = tens(part["train"]); val = tens(part["val"]); test = part["test"]
gx, gy = np.meshgrid(truth["x"], truth["y"], indexing="ij")

demo, hist = {}, {}
for name, wp in (("DataOnlyNN", 0.0), ("PINN", 0.01)):
    m = twin_model(tw)
    t0 = time.time()
    hist[name] = train(m, Xtr, ctr, TrainConfig(n_iters=1500, seed=SEED, w_p=wp, w_b=0.1 if wp else 0.0,
                                                w_s=1e-3 if wp else 0.0, log_every=50), val=val)
    with torch.no_grad():
        p = m(sc.inputs(test.x * 1e3, test.y * 1e3, test.t_rel)).numpy() * sc.C_ref
        F = np.stack([m(sc.inputs(gx.ravel(), gy.ravel(), np.full(gx.size, tt))).numpy() * sc.C_ref
                      for tt in truth["t_rel"][::6]])
    field_rmse = np.sqrt(np.mean((F - truth["C"][::6].reshape(len(F), -1)) ** 2))
    demo[name] = {**{k_: round(v, 3) for k_, v in evaluate(test.pm25, p).items() if k_ in ("mae", "rmse", "r2")},
                  "field_rmse": round(float(field_rmse), 2), "D": round(m.D().item(), 1), "seconds": round(time.time() - t0)}
res = pd.DataFrame(demo).T
better = lambda col: "PINN" if res.loc["PINN", col] < res.loc["DataOnlyNN", col] else "data-only NN"
print(f"Lower whole-field RMSE: {better('field_rmse')} | lower held-out-station RMSE: {better('rmse')}")
res

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.2))
for name, h in hist.items():
    it = [r["iter"] for r in h]
    axs[0].semilogy(it, [r["data"] for r in h], color=COL[name], label=f"{name}: train data loss")
    axs[0].semilogy(it, [r["val"] for r in h], color=COL[name], ls="--", label=f"{name}: validation MSE")
for key, ls in (("data", "-"), ("physics", "--"), ("boundary", ":"), ("source_reg", "-.")):
    axs[1].semilogy([r["iter"] for r in hist["PINN"]], [r[key] for r in hist["PINN"]], "k", ls=ls, label=key)
axs[0].set(xlabel="iteration", ylabel="dimensionless MSE", title="Data fit"); axs[0].legend(fontsize=7)
axs[1].set(xlabel="iteration", ylabel="unweighted loss", title="PINN loss components"); axs[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

**Interpretation.** In this shortened run the PINN already has the lower **whole-field** error (scored against the known
truth on the grid): the PDE carries information from the stations along the wind into the gaps between them. At the
**held-out stations** it may not yet be ahead — the PINN optimises a harder, multi-term objective and converges more
slowly (its data loss is still falling at 1500 iterations, while the data-only network has nearly fitted its stations).
This is why the study uses 6000 iterations; with the full schedule (below) the PINN is also better at the held-out stations
on average. The demonstration shows the mechanism; the full study is the reference for the numbers.

### 4.3 Full synthetic-twin results (loaded; 4 station splits, 6000 iterations)

In [ ]:
twin = read_csv("results/metrics/twin_seed_summary.csv")
twin.groupby("model")[["rmse", "mae", "field"]].agg(["mean", "std"]).round(1).loc[["IDW", "RandomForest", "DataOnlyNN", "PINN"]]

In [ ]:
for f in ("twin_comparison", "twin_fields", "twin_residual", "twin_identifiability"):
    show_figure(f)

**Interpretation (SYNTHETIC).**
* The PINN has the lowest held-out error and the smallest spread across splits; over the whole field it is about 43 % more
  accurate than the identical network without physics.
* Its fields satisfy the PDE about 4× better, even after fitting an emission model to the data-only field — station fit
  alone does not produce a transport-consistent field.
* **Identifiability:** $D$ was initialised at its true value (300 m²/s) yet converges to ≈ 60 m²/s in every split,
  exactly as the Péclet-number analysis predicts; the diurnal emission profile, by contrast, is recovered well.

## 5. Observed data: held-out Delhi monitors

**Protocol.** Stations (not hours) are split 70/15/15 into train/validation/test, with co-located stations (< 1 km) kept
together; leakage is asserted in code; all models share the split. Each 10-day window is trained as three 80-hour chunks
(time-domain decomposition, to keep $U$ comparable to the twin). $w_p=0.01$ was **fixed from the twin before** any real-data
run. Baselines: hourly mean of the training stations (a spatially uniform field), inverse-distance weighting, random forest.

In [ ]:
real = read_csv("results/metrics/real_seed_summary.csv")
order = ["StationMean", "IDW", "RandomForest", "DataOnlyNN", "PINN"]
tab = real.groupby(["window", "model"]).rmse.agg(["mean", "std"]).round(1).unstack(0)
tab.loc[order]

In [ ]:
wins = real.pivot_table(index=["window", "seed"], columns="model", values="rmse")
print("PINN better than data-only NN in", int((wins.PINN < wins.DataOnlyNN).sum()), "of", len(wins), "runs")
print("PINN best of all five models in", int((wins[order].idxmin(axis=1) == "PINN").sum()), "of", len(wins), "runs")
for f in ("real_comparison", "real_station_series", "real_residual", "real_wind_check"):
    show_figure(f)

**Interpretation (OBSERVED).**
* **Physics regularises.** The PINN beats the identical data-only network in every run and is far more stable across
  splits; its fields have a 4–8× smaller PDE residual.
* **It does not beat simple baselines.** In winter the PINN, IDW and the station mean are within a few µg/m³ — much less than
  the split-to-split spread — and in spring the station mean is best. This is an honest negative result.
* **Why.** The twin shows the method works when the model is right, so the gap is model mismatch: a strong city-wide common
  signal (r ≈ 0.7), uncertain transport wind (monitor anemometers correlate with ERA5 at only r ≈ 0.5), no regional inflow,
  no entrainment and no secondary aerosol formation. The learned $D$ again varies widely (37–266 m²/s) between chunks.

## 6. Sensitivity analysis

Two questions: how sensitive is the **physical model** to its parameters (forward sensitivity, reference solver), and how
sensitive is the **PINN** to the physics weight $w_p$?

The elasticity $\varepsilon_p = \partial \ln \bar C / \partial \ln p$ is estimated from the end points of each sweep.

In [ ]:
sens = read_csv("results/metrics/twin_forward_sensitivity.csv")
el = {}
for p, d in sens.groupby("param"):
    d = d.sort_values("value")
    el[p] = np.log(d["mean"].iloc[-1] / d["mean"].iloc[0]) / np.log(d.value.iloc[-1] / d.value.iloc[0])
print("elasticity of station-mean concentration:", {k_: round(v, 3) for k_, v in el.items()})
for f in ("twin_forward_sensitivity", "twin_wp_sweep", "real_wp_sweep"):
    show_figure(f)

**Interpretation.** Station concentrations are almost insensitive to $D$ (elasticity ≈ −0.03), mildly sensitive to $\lambda$,
and most sensitive to the mixing-height floor $H_{\min}$ — a modelling choice, which we flag as a limitation. For the PINN,
$w_p$ around 0.003–0.01 is best on the twin; at $w_p \ge 0.1$ the PDE term dominates the gradient and the network under-fits
the data, a known PINN pathology (gradient imbalance). On real data the conclusion is unchanged across $w_p\in[0.003,0.03]$.

## 7. Conclusions and limitations

* When its assumptions hold, the physics constraint improves generalisation to unseen locations, especially away from stations,
  and yields physically consistent fields.
* On observed Delhi data the physics still stabilises the network and enforces consistency, but the simplified model does not
  outperform simple interpolation; the main limitations are the uniform wind, unknown inflow boundary, missing entrainment and
  chemistry, and a strong city-wide common signal.
* The eddy diffusivity is not identifiable at Pe ~ 10²–10³; the diurnal emission profile is.

**How the stored results were produced.** The full-length runs take several hours of CPU time, so their result tables and figures
are embedded rather than recomputed. The model and training code in §4 is the code used for the synthetic-twin study: to repeat one
full twin run, set `n_iters=6000` in §4.2 and pick a split with `twin_setup(seed)` (the study used seeds 42, 1, 2 and 3; §4.2 scores
the whole field every 6 h, the study every 3 h). The observed-data runs in §5 used the same trainer on three 80-hour chunks per window;
that driver and the raw-data download and cleaning steps are not part of this notebook, which starts from the cleaned tables.